
# Task 3 — Drift-aware Federated Learning on MNIST

This notebook implements:
- **Algorithm 4 style drift-aware clustering** inspired by the paper
- **Random**, **DivFL-like**, and **S-FedAvg-like** client selection baselines
- **Sudden label drift** with two drift events
- **Sudden feature drift** with two drift events
- **Resumable training** with checkpoints saved to Google Drive or local storage

The training can continue from the latest saved checkpoint after disconnects on CPU or GPU.


In [1]:

# =========================
# 1. Imports, seeds, and storage
# =========================

import os
import json
import math
import random
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display

from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

# -------------------------
# Reproducibility
# -------------------------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# -------------------------
# Optional Google Drive mount
# -------------------------
IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount skipped:", e)

# -------------------------
# Storage root
# -------------------------
if IN_COLAB and Path("/content/drive/MyDrive").exists():
    BASE_DIR = Path("/content/drive/MyDrive/Task3_FedDrift_MNIST")
else:
    BASE_DIR = Path("/mnt/data/Task3_FedDrift_MNIST")

BASE_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR =", BASE_DIR)

# -------------------------
# Federated learning config
# -------------------------
N_CLIENTS = 100
ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 32
CANDIDATE_POOL_SIZE = 30
WARMUP_ROUNDS = 3
CLIENT_ALPHA = 0.4
MAX_CLIENT_SAMPLES_PER_CLIENT = 600

K_VALUES = [10]  # vary K as required by the assignment

# Drift scenarios and drift points
SCENARIOS = ["label_drift", "feature_drift"]
LABEL_DRIFT_ROUNDS = (30, 70)
FEATURE_DRIFT_ROUNDS = (30, 70)

NUM_CLASSES = 10
IMAGE_SHAPE = (28, 28, 1)

# thresholds used by the drift-aware clustering
DRIFT_DELTA = 0.20
CLUSTER_MERGE_THRESHOLD = 0.25

# evaluation sample caps
MAX_CLUSTER_EVAL_SAMPLES = 512
MAX_CLIENT_EVAL_SAMPLES = 256

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/Task3_FedDrift_MNIST
TensorFlow: 2.19.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:

# =========================
# 2. MNIST loading and non-IID client partitioning
# =========================

(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train_full = y_train_full.astype(np.int64)
y_test = y_test.astype(np.int64)

x_train_full = np.expand_dims(x_train_full, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Train shape:", x_train_full.shape, y_train_full.shape)
print("Test shape :", x_test.shape, y_test.shape)

def dirichlet_partition(labels, n_clients=100, alpha=0.4, seed=42, max_samples_per_client=600):
    """
    Non-IID split using a Dirichlet distribution over labels.
    Each client receives a skewed label distribution.
    """
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    n_classes = len(np.unique(labels))
    client_indices = {i: [] for i in range(n_clients)}

    for c in range(n_classes):
        idx_c = np.where(labels == c)[0]
        rng.shuffle(idx_c)

        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        counts = np.floor(proportions * len(idx_c)).astype(int)

        # adjust counts so that they sum exactly
        diff = len(idx_c) - counts.sum()
        for _ in range(abs(diff)):
            j = rng.integers(0, n_clients)
            counts[j] += 1 if diff > 0 else -1

        start = 0
        for client_id, cnt in enumerate(counts):
            if cnt <= 0:
                continue
            end = start + cnt
            client_indices[client_id].extend(idx_c[start:end].tolist())
            start = end

    # optionally cap the number of samples per client
    for cid in range(n_clients):
        idxs = np.array(client_indices[cid], dtype=int)
        if len(idxs) == 0:
            continue
        rng.shuffle(idxs)
        if max_samples_per_client is not None and len(idxs) > max_samples_per_client:
            idxs = idxs[:max_samples_per_client]
        client_indices[cid] = idxs.tolist()

    return client_indices

client_train_indices = dirichlet_partition(
    y_train_full,
    n_clients=N_CLIENTS,
    alpha=CLIENT_ALPHA,
    seed=SEED,
    max_samples_per_client=MAX_CLIENT_SAMPLES_PER_CLIENT,
)

# Split each client's local data into train / validation
client_data = {}
for cid, idxs in client_train_indices.items():
    idxs = np.array(idxs, dtype=int)
    rng = np.random.default_rng(SEED + cid)
    rng.shuffle(idxs)

    if len(idxs) == 0:
        client_data[cid] = {
            "x_train": np.empty((0, 28, 28, 1), dtype=np.float32),
            "y_train": np.empty((0,), dtype=np.int64),
            "x_val": np.empty((0, 28, 28, 1), dtype=np.float32),
            "y_val": np.empty((0,), dtype=np.int64),
        }
        continue

    val_size = max(10, int(0.2 * len(idxs)))
    val_size = min(val_size, len(idxs) // 2 if len(idxs) >= 2 else len(idxs))

    val_idx = idxs[:val_size]
    train_idx = idxs[val_size:]

    client_data[cid] = {
        "x_train": x_train_full[train_idx],
        "y_train": y_train_full[train_idx],
        "x_val": x_train_full[val_idx],
        "y_val": y_train_full[val_idx],
    }

# Quick stats
client_sizes = [len(client_data[cid]["x_train"]) for cid in range(N_CLIENTS)]
print("Clients with data:", sum(s > 0 for s in client_sizes))
print("Avg train samples/client:", float(np.mean([s for s in client_sizes if s > 0])))
print("Min/Max train samples/client:", int(np.min([s for s in client_sizes if s > 0])), int(np.max(client_sizes)))


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train shape: (60000, 28, 28, 1) (60000,)
Test shape : (10000, 28, 28, 1) (10000,)
Clients with data: 100
Avg train samples/client: 399.98
Min/Max train samples/client: 96 480


In [3]:

# =========================
# 3. Sudden drift schedules (2 events for label, 2 events for feature)
# =========================

# 4 cohorts of clients
client_order = np.random.default_rng(SEED).permutation(N_CLIENTS)
client_groups = np.array_split(client_order, 4)
client_group_id = {}
for gid, group in enumerate(client_groups):
    for cid in group:
        client_group_id[int(cid)] = gid

# Two label-drift events and two feature-drift events:
# groups 0-1 change at round 30, groups 2-3 change at round 70
EARLY_GROUPS = {0, 1}
LATE_GROUPS = {2, 3}

def compose_label_maps(*maps):
    composed = {}
    for i in range(10):
        value = i
        for mapping in maps:
            value = mapping.get(value, value)
        composed[i] = value
    return composed

# Two strong label permutations
LABEL_MAP_1 = {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 7, 7: 8, 8: 9, 9: 0}
LABEL_MAP_2 = {0: 9, 9: 8, 8: 7, 7: 6, 6: 5, 5: 4, 4: 3, 3: 2, 2: 1, 1: 0}

def apply_label_map(y, label_map):
    if len(y) == 0:
        return y
    y_new = np.array([label_map.get(int(v), int(v)) for v in y], dtype=np.int64)
    return y_new

def invert_images(x):
    return np.clip(1.0 - x, 0.0, 1.0)

def rotate_images_90(x):
    if len(x) == 0:
        return x
    rotated = np.stack([np.rot90(img.squeeze(-1), k=1)[..., None] for img in x], axis=0).astype(np.float32)
    return rotated

def shift_images(x, shift=(2, 2)):
    if len(x) == 0:
        return x
    sx, sy = shift
    out = np.zeros_like(x)
    out[:, sx:, sy:, :] = x[:, :-sx, :-sy, :]
    return out

def maybe_apply_label_drift(cid, round_idx, x, y):
    gid = client_group_id[cid]
    if gid in EARLY_GROUPS and round_idx >= LABEL_DRIFT_ROUNDS[0]:
        return x, apply_label_map(y, LABEL_MAP_1)
    if gid in LATE_GROUPS and round_idx >= LABEL_DRIFT_ROUNDS[1]:
        return x, apply_label_map(y, LABEL_MAP_2)
    return x, y

def maybe_apply_feature_drift(cid, round_idx, x, y):
    gid = client_group_id[cid]
    if gid in EARLY_GROUPS and round_idx >= FEATURE_DRIFT_ROUNDS[0]:
        return invert_images(x), y
    if gid in LATE_GROUPS and round_idx >= FEATURE_DRIFT_ROUNDS[1]:
        return rotate_images_90(x), y
    return x, y

def get_client_round_data(cid, round_idx, scenario):
    """
    Returns drifted client train/val data for a specific round and scenario.
    """
    x_train = client_data[cid]["x_train"].copy()
    y_train = client_data[cid]["y_train"].copy()
    x_val = client_data[cid]["x_val"].copy()
    y_val = client_data[cid]["y_val"].copy()

    if scenario == "label_drift":
        x_train, y_train = maybe_apply_label_drift(cid, round_idx, x_train, y_train)
        x_val, y_val = maybe_apply_label_drift(cid, round_idx, x_val, y_val)
    elif scenario == "feature_drift":
        x_train, y_train = maybe_apply_feature_drift(cid, round_idx, x_train, y_train)
        x_val, y_val = maybe_apply_feature_drift(cid, round_idx, x_val, y_val)
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    return x_train, y_train, x_val, y_val

def get_drifted_eval_set(round_idx, scenario, per_client_sample=MAX_CLIENT_EVAL_SAMPLES):
    """
    Build an evaluation set using the current drift state of all clients.
    """
    xs, ys = [], []
    rng = np.random.default_rng(SEED + 999 + round_idx)

    for cid in range(N_CLIENTS):
        _, _, x_val, y_val = get_client_round_data(cid, round_idx, scenario)
        if len(x_val) == 0:
            continue
        if len(x_val) > per_client_sample:
            idx = rng.choice(len(x_val), size=per_client_sample, replace=False)
            x_val = x_val[idx]
            y_val = y_val[idx]
        xs.append(x_val)
        ys.append(y_val)

    if not xs:
        return np.empty((0, 28, 28, 1), dtype=np.float32), np.empty((0,), dtype=np.int64)

    return np.concatenate(xs, axis=0), np.concatenate(ys, axis=0)

print("Example cohort sizes:", [len(g) for g in client_groups])
print("Label drift rounds:", LABEL_DRIFT_ROUNDS)
print("Feature drift rounds:", FEATURE_DRIFT_ROUNDS)


Example cohort sizes: [25, 25, 25, 25]
Label drift rounds: (30, 70)
Feature drift rounds: (30, 70)


In [4]:

# =========================
# 4. CNN model and utility helpers
# =========================

def build_cnn_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=IMAGE_SHAPE),
        tf.keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def clone_model_with_weights(model):
    cloned = tf.keras.models.clone_model(model)
    cloned.build((None,) + IMAGE_SHAPE)
    cloned.set_weights(model.get_weights())
    cloned.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return cloned

def model_size_bytes(model):
    n_params = np.sum([np.prod(w.shape) for w in model.get_weights()])
    return int(n_params * 4)

def weighted_average_weights(weight_list, sample_counts):
    total = float(np.sum(sample_counts)) + 1e-12
    avg_weights = []
    for layer_weights in zip(*weight_list):
        layer_sum = np.zeros_like(layer_weights[0], dtype=np.float32)
        for w, n in zip(layer_weights, sample_counts):
            layer_sum += w * (n / total)
        avg_weights.append(layer_sum)
    return avg_weights

def sample_xy(x, y, max_samples, seed):
    if len(x) <= max_samples:
        return x, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(x), size=max_samples, replace=False)
    return x[idx], y[idx]

def evaluate_model(model, x, y):
    if len(x) == 0:
        return float("nan"), float("nan")
    loss, acc = model.evaluate(x, y, batch_size=256, verbose=0)
    return float(loss), float(acc)

def softmax_predict(model, x):
    return model.predict(x, batch_size=256, verbose=0)

def first_round_at_least(values, threshold=0.80):
    for i, v in enumerate(values, start=1):
        if v >= threshold:
            return i
    return None


In [5]:

# =========================
# 5. Client state and checkpoint helpers
# =========================

def initialize_client_state(n_clients):
    state = {}
    for cid in range(n_clients):
        state[cid] = {
            "ema_loss": None,
            "prev_loss": None,
            "drift_score": 0.0,
            "proxy_score": 0.0,
            "last_update_embedding": np.zeros(4, dtype=np.float32),
            "last_participated_round": -1,
            "participation_count": 0,
            "cluster_id": -1,
            "prev_best_model_loss": None,
            "last_drift_round": -1,
        }
    return state

def update_client_state(client_state, client_id, pre_loss, post_loss, update_embedding, round_idx):
    entry = client_state[client_id]
    alpha = 0.7

    entry["prev_loss"] = entry["ema_loss"]
    if entry["ema_loss"] is None:
        entry["ema_loss"] = float(post_loss)
    else:
        entry["ema_loss"] = alpha * float(post_loss) + (1.0 - alpha) * entry["ema_loss"]

    if entry["prev_loss"] is None:
        entry["drift_score"] = 0.0
    else:
        entry["drift_score"] = float(post_loss - entry["prev_loss"])

    entry["proxy_score"] = float(max(0.0, pre_loss - post_loss))
    entry["last_update_embedding"] = np.asarray(update_embedding, dtype=np.float32)
    entry["last_participated_round"] = round_idx
    entry["participation_count"] += 1

def layer_update_embedding(before_weights, after_weights):
    return np.asarray([np.linalg.norm(a - b) for a, b in zip(after_weights, before_weights)], dtype=np.float32)

def save_json(path, payload):
    path = Path(path)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "r") as f:
        return json.load(f)

def save_pickle(path, payload):
    path = Path(path)
    with open(path, "wb") as f:
        pickle.dump(payload, f)

def load_pickle(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, "rb") as f:
        return pickle.load(f)

def save_model_weights(model, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model.save_weights(str(path))

def load_model_weights_into(model, path):
    path = Path(path)
    model.load_weights(str(path))

def make_history():
    return {
        "round": [],
        "clean_accuracy": [],
        "drifted_accuracy": [],
        "communication_bytes": [],
        "computation_samples": [],
        "selected_clients": [],
        "cluster_count": [],
        "drifted_clients_count": [],
        "mean_drift_score": [],
        "mean_proxy_score": [],
    }

def history_summary_row(strategy, k, scenario, hist):
    return {
        "strategy": strategy,
        "K": k,
        "scenario": scenario,
        "final_clean_accuracy": float(hist["clean_accuracy"][-1]) if hist["clean_accuracy"] else float("nan"),
        "final_drifted_accuracy": float(hist["drifted_accuracy"][-1]) if hist["drifted_accuracy"] else float("nan"),
        "best_clean_accuracy": float(np.max(hist["clean_accuracy"])) if hist["clean_accuracy"] else float("nan"),
        "best_drifted_accuracy": float(np.max(hist["drifted_accuracy"])) if hist["drifted_accuracy"] else float("nan"),
        "round_to_80pct_clean": first_round_at_least(hist["clean_accuracy"], 0.80),
        "round_to_80pct_drifted": first_round_at_least(hist["drifted_accuracy"], 0.80),
        "total_communication_MB": float(np.sum(hist["communication_bytes"]) / (1024**2)),
        "total_computation_samples": int(np.sum(hist["computation_samples"])) if hist["computation_samples"] else 0,
        "final_cluster_count": int(hist["cluster_count"][-1]) if hist["cluster_count"] else 1,
    }


In [6]:

# =========================
# 6. Baseline client selection strategies
# =========================

def random_select(candidate_clients, k):
    rng = np.random.default_rng(SEED + len(candidate_clients) + k)
    k = min(k, len(candidate_clients))
    return rng.choice(candidate_clients, size=k, replace=False).tolist()

def divfl_like_select(candidate_clients, k, client_state):
    candidate_clients = list(candidate_clients)
    if len(candidate_clients) <= k:
        return candidate_clients

    # seed client: strongest update magnitude
    seed = max(candidate_clients, key=lambda c: np.linalg.norm(client_state[c]["last_update_embedding"]) + 1e-6)
    selected = [seed]
    remaining = [c for c in candidate_clients if c != seed]

    while len(selected) < k and remaining:
        best_client = None
        best_score = -1e18

        selected_embeddings = [client_state[s]["last_update_embedding"] for s in selected]
        selected_mean = np.mean(selected_embeddings, axis=0)

        for cid in remaining:
            emb = client_state[cid]["last_update_embedding"]
            diversity = float(np.linalg.norm(emb - selected_mean))
            utility = float(client_state[cid]["proxy_score"])
            freshness = 1.0 / (1.0 + max(0, len(candidate_clients) - client_state[cid]["last_participated_round"]))
            score = 0.55 * diversity + 0.30 * utility + 0.15 * freshness
            if score > best_score:
                best_score = score
                best_client = cid

        selected.append(best_client)
        remaining.remove(best_client)

    return selected

def sfedavg_like_select(candidate_clients, k, client_state):
    candidate_clients = list(candidate_clients)
    if len(candidate_clients) <= k:
        return candidate_clients

    scores = []
    for cid in candidate_clients:
        st = client_state[cid]
        recency = 1.0 / (1.0 + max(0, st["last_participated_round"]))
        score = 0.55 * st["proxy_score"] - 0.30 * abs(st["drift_score"]) + 0.15 * recency
        scores.append((score, cid))

    scores.sort(reverse=True, key=lambda t: t[0])
    return [cid for _, cid in scores[:k]]

def select_clients(strategy, candidate_clients, k, client_state):
    if len(candidate_clients) <= k:
        return list(candidate_clients)

    if strategy == "random":
        return random_select(candidate_clients, k)
    if strategy == "divfl":
        return divfl_like_select(candidate_clients, k, client_state)
    if strategy == "sfedavg":
        return sfedavg_like_select(candidate_clients, k, client_state)

    raise ValueError(f"Unknown baseline strategy: {strategy}")


In [7]:

# =========================
# 7. Algorithm 4 style drift-aware clustering and selection
# =========================

def initialize_feddrift_state():
    base_model = build_cnn_model()
    return {
        "models": {0: base_model},
        "cluster_members": {0: set(range(N_CLIENTS))},
        "client_to_model": {cid: 0 for cid in range(N_CLIENTS)},
        "next_model_id": 1,
    }

def collect_cluster_eval_data(cluster_member_ids, round_cache, round_idx, max_samples=MAX_CLUSTER_EVAL_SAMPLES):
    xs, ys = [], []
    for cid in cluster_member_ids:
        _, _, x_val, y_val = round_cache[cid]
        if len(x_val) > 0:
            xs.append(x_val)
            ys.append(y_val)

    if not xs:
        return None, None

    x = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    x, y = sample_xy(x, y, max_samples=max_samples, seed=SEED + 999 + round_idx + len(cluster_member_ids))
    return x, y

def evaluate_models_for_client(state, cid, round_cache):
    """
    Evaluate every current model on this client's current validation data.
    """
    _, _, x_val, y_val = round_cache[cid]
    losses = {}
    for mid, model in state["models"].items():
        loss, _ = evaluate_model(model, x_val, y_val)
        losses[mid] = loss
    return losses

def merge_models(state, mid_a, mid_b, round_cache):
    members_a = state["cluster_members"][mid_a]
    members_b = state["cluster_members"][mid_b]
    merged_members = members_a | members_b

    size_a = sum(round_cache[cid][0].shape[0] for cid in members_a)
    size_b = sum(round_cache[cid][0].shape[0] for cid in members_b)

    weights_a = state["models"][mid_a].get_weights()
    weights_b = state["models"][mid_b].get_weights()
    merged_weights = weighted_average_weights([weights_a, weights_b], [size_a, size_b])

    new_mid = state["next_model_id"]
    state["next_model_id"] += 1

    new_model = clone_model_with_weights(state["models"][mid_a])
    new_model.set_weights(merged_weights)

    del state["models"][mid_a]
    del state["models"][mid_b]
    del state["cluster_members"][mid_a]
    del state["cluster_members"][mid_b]

    state["models"][new_mid] = new_model
    state["cluster_members"][new_mid] = merged_members

    for cid in merged_members:
        state["client_to_model"][cid] = new_mid

    return new_mid


def ensure_cluster_keys_exist(state):
    """
    Repairs the runtime state so every model id has a cluster entry.
    This prevents KeyError when a model exists but cluster_members[mid] is missing.
    """
    if "cluster_members" not in state:
        state["cluster_members"] = {}

    if "client_to_model" not in state:
        state["client_to_model"] = {}

    for mid in list(state["models"].keys()):
        state["cluster_members"].setdefault(mid, set())

    # normalize keys/types
    state["cluster_members"] = {
        int(mid): set(members) for mid, members in state["cluster_members"].items()
    }
    state["client_to_model"] = {
        int(cid): int(mid) for cid, mid in state["client_to_model"].items()
    }
    return state

def feddrift_assign_and_merge(state, client_state, round_cache, round_idx):
    """
    Algorithm 4:
    1) compute losses for all clients against all models
    2) detect drift locally
    3) create singleton models for drifted clients
    4) hierarchical merge of close clusters
    """
    state = ensure_cluster_keys_exist(state)
    client_best = {}
    drifted_clients = []

    # local drift detection
    for cid in range(N_CLIENTS):
        loss_map = evaluate_models_for_client(state, cid, round_cache)
        best_mid, best_loss = min(loss_map.items(), key=lambda kv: kv[1])
        client_best[cid] = (best_mid, best_loss)

        prev_best = client_state[cid]["prev_best_model_loss"]
        if prev_best is not None and best_loss > prev_best + DRIFT_DELTA:
            drifted_clients.append(cid)

        client_state[cid]["prev_best_model_loss"] = best_loss

    # create singleton models for drifted clients
    new_assignments = {}
    for cid in range(N_CLIENTS):
        best_mid, best_loss = client_best[cid]
        if cid in drifted_clients:
            new_mid = state["next_model_id"]
            state["next_model_id"] += 1
            new_model = clone_model_with_weights(state["models"][best_mid])
            state["models"][new_mid] = new_model
            state["cluster_members"][new_mid] = {cid}
            new_assignments[cid] = new_mid
            client_state[cid]["last_drift_round"] = round_idx
        else:
            new_assignments[cid] = best_mid

    # rebuild cluster membership from assignments
    rebuilt_members = defaultdict(set)
    for cid, mid in new_assignments.items():
        rebuilt_members[mid].add(cid)

    # remove models that ended up with no clients
    valid_model_ids = {mid for mid, members in rebuilt_members.items() if len(members) > 0}
    state["models"] = {mid: model for mid, model in state["models"].items() if mid in valid_model_ids}
    state["cluster_members"] = {mid: members for mid, members in rebuilt_members.items() if len(members) > 0}
    state["client_to_model"] = {cid: mid for cid, mid in new_assignments.items() if mid in valid_model_ids}

    state = ensure_cluster_keys_exist(state)

    # hierarchical merging
    while True:
        mids = list(state["models"].keys())
        if len(mids) < 2:
            break

        cluster_data = {}
        for mid in mids:
            x_c, y_c = collect_cluster_eval_data(
                state["cluster_members"][mid],
                round_cache,
                round_idx,
                max_samples=MAX_CLUSTER_EVAL_SAMPLES
            )
            cluster_data[mid] = (x_c, y_c)

        best_pair = None
        best_dist = float("inf")

        for i in range(len(mids)):
            for j in range(i + 1, len(mids)):
                mi, mj = mids[i], mids[j]
                xi, yi = cluster_data[mi]
                xj, yj = cluster_data[mj]

                if xi is None or xj is None:
                    continue

                Lii, _ = evaluate_model(state["models"][mi], xi, yi)
                Ljj, _ = evaluate_model(state["models"][mj], xj, yj)
                Lij, _ = evaluate_model(state["models"][mi], xj, yj)
                Lji, _ = evaluate_model(state["models"][mj], xi, yi)

                D = max(Lij - Lii, Lji - Ljj, 0.0)

                if D < best_dist:
                    best_dist = D
                    best_pair = (mi, mj)

        if best_pair is None or best_dist >= CLUSTER_MERGE_THRESHOLD:
            break

        merge_models(state, best_pair[0], best_pair[1], round_cache)

    # final clean-up of the mapping
    updated_mapping = {}
    for mid, members in state["cluster_members"].items():
        for cid in members:
            updated_mapping[cid] = mid
            client_state[cid]["cluster_id"] = mid

    state["client_to_model"] = updated_mapping

    return state, drifted_clients

def cluster_selection_score(member_ids, client_state, round_idx):
    if not member_ids:
        return -1e18
    mean_proxy = float(np.mean([client_state[c]["proxy_score"] for c in member_ids]))
    mean_drift = float(np.mean([client_state[c]["drift_score"] for c in member_ids]))
    size_term = math.log1p(len(member_ids))
    recency = float(np.mean([1.0 / (1.0 + max(0, round_idx - client_state[c]["last_participated_round"])) for c in member_ids]))
    return 0.50 * mean_proxy + 0.25 * abs(mean_drift) + 0.15 * size_term + 0.10 * recency

def client_selection_score(cid, client_state, round_idx):
    st = client_state[cid]
    recency = 1.0 / (1.0 + max(0, round_idx - st["last_participated_round"]))
    novelty = 1.0 if st["last_drift_round"] == round_idx else 0.0
    return (
        0.55 * st["proxy_score"] +
        0.20 * abs(st["drift_score"]) +
        0.15 * recency +
        0.10 * novelty
    )

def select_k_clients_from_clusters(state, client_state, candidate_pool, k, round_idx):
    """
    Choose K clients from the candidate pool using cluster-aware ranking.
    """
    candidate_pool = list(candidate_pool)
    if len(candidate_pool) <= k:
        return candidate_pool

    cluster_to_candidates = defaultdict(list)
    for cid in candidate_pool:
        mid = state["client_to_model"].get(cid, 0)
        cluster_to_candidates[mid].append(cid)

    cluster_ranking = sorted(
        cluster_to_candidates.items(),
        key=lambda item: cluster_selection_score(item[1], client_state, round_idx),
        reverse=True,
    )

    selected = []
    used = set()

    # pick one representative from strong clusters first
    for mid, members in cluster_ranking:
        rep = max(members, key=lambda c: client_selection_score(c, client_state, round_idx))
        if rep not in used:
            selected.append(rep)
            used.add(rep)
        if len(selected) == k:
            return selected

    # fill remaining slots with the best remaining candidates
    remaining = [cid for cid in candidate_pool if cid not in used]
    remaining.sort(key=lambda c: client_selection_score(c, client_state, round_idx), reverse=True)

    for cid in remaining:
        selected.append(cid)
        if len(selected) == k:
            break

    return selected

def evaluate_feddrift_ensemble(state, x, y):
    mids = list(state["models"].keys())
    if not mids or len(x) == 0:
        return float("nan"), float("nan")

    cluster_sizes = np.array([len(state["cluster_members"].get(mid, [])) for mid in mids], dtype=np.float32)
    cluster_weights = cluster_sizes / (np.sum(cluster_sizes) + 1e-12)

    probs = np.zeros((len(x), NUM_CLASSES), dtype=np.float32)
    for w, mid in zip(cluster_weights, mids):
        p = state["models"][mid].predict(x, batch_size=256, verbose=0)
        probs += w * p

    preds = np.argmax(probs, axis=1)
    acc = float(np.mean(preds == y))
    eps = 1e-12
    loss = float(-np.mean(np.log(np.clip(probs[np.arange(len(y)), y], eps, 1.0))))
    return loss, acc


In [8]:

# =========================
# 8. Training functions
# =========================

def train_selected_clients_single_model(global_model, selected_clients, round_cache, client_state, round_idx):
    """
    FedAvg with a single global model (used by random, DivFL-like, and S-FedAvg-like baselines).
    """
    local_weights = []
    sample_counts = []
    communication_bytes = 0
    computation_samples = 0

    for cid in selected_clients:
        x_train, y_train, x_val, y_val = round_cache[cid]
        if len(x_train) == 0 or len(x_val) == 0:
            continue

        local_model = clone_model_with_weights(global_model)
        before_weights = [w.copy() for w in local_model.get_weights()]

        pre_loss, _ = local_model.evaluate(x_val, y_val, batch_size=BATCH_SIZE, verbose=0)

        local_model.fit(
            x_train,
            y_train,
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0,
            shuffle=True,
        )

        post_loss, _ = local_model.evaluate(x_val, y_val, batch_size=BATCH_SIZE, verbose=0)
        after_weights = local_model.get_weights()
        update_emb = layer_update_embedding(before_weights, after_weights)

        update_client_state(client_state, cid, pre_loss, post_loss, update_emb, round_idx)

        local_weights.append(after_weights)
        sample_counts.append(len(x_train))
        computation_samples += int(len(x_train) * LOCAL_EPOCHS)
        communication_bytes += 2 * model_size_bytes(global_model)

    if local_weights:
        global_model.set_weights(weighted_average_weights(local_weights, sample_counts))

    return global_model, communication_bytes, computation_samples

def train_selected_clients_per_cluster(state, selected_clients, round_cache, client_state, round_idx):
    """
    Per-cluster FedAvg for the Algorithm-4-style multi-model method.
    """
    selected_by_model = defaultdict(list)
    for cid in selected_clients:
        mid = state["client_to_model"].get(cid, 0)
        if mid in state["models"]:
            selected_by_model[mid].append(cid)

    communication_bytes = 0
    computation_samples = 0

    for mid, cids in selected_by_model.items():
        base_model = clone_model_with_weights(state["models"][mid])
        local_weights = []
        sample_counts = []

        for cid in cids:
            x_train, y_train, x_val, y_val = round_cache[cid]
            if len(x_train) == 0 or len(x_val) == 0:
                continue

            local_model = clone_model_with_weights(base_model)
            before_weights = [w.copy() for w in local_model.get_weights()]

            pre_loss, _ = local_model.evaluate(x_val, y_val, batch_size=BATCH_SIZE, verbose=0)

            local_model.fit(
                x_train,
                y_train,
                epochs=LOCAL_EPOCHS,
                batch_size=BATCH_SIZE,
                verbose=0,
                shuffle=True,
            )

            post_loss, _ = local_model.evaluate(x_val, y_val, batch_size=BATCH_SIZE, verbose=0)
            after_weights = local_model.get_weights()
            update_emb = layer_update_embedding(before_weights, after_weights)

            update_client_state(client_state, cid, pre_loss, post_loss, update_emb, round_idx)

            local_weights.append(after_weights)
            sample_counts.append(len(x_train))
            computation_samples += int(len(x_train) * LOCAL_EPOCHS)
            communication_bytes += 2 * model_size_bytes(base_model)

        if local_weights:
            state["models"][mid].set_weights(weighted_average_weights(local_weights, sample_counts))

    return state, communication_bytes, computation_samples


In [9]:
# =========================
# 9. Checkpointing and resumable experiment directories
# =========================

def experiment_dir(strategy, k, scenario):
    d = BASE_DIR / f"{strategy}__K{k}__{scenario}"
    d.mkdir(parents=True, exist_ok=True)
    (d / "models").mkdir(parents=True, exist_ok=True)
    return d

def baseline_checkpoint_path(run_dir):
    return run_dir / "checkpoint.pkl"

def algo4_checkpoint_path(run_dir):
    return run_dir / "checkpoint.pkl"

def save_baseline_checkpoint(run_dir, round_idx, global_model, client_state, history):
    payload = {
        "round_idx": round_idx,
        "client_state": client_state,
        "history": history,
        "model_weights_path": str(run_dir / "global_model.weights.h5"),
    }
    save_pickle(baseline_checkpoint_path(run_dir), payload)
    save_model_weights(global_model, run_dir / "global_model.weights.h5")
    save_json(run_dir / "history.json", history)

def load_baseline_checkpoint(run_dir):
    ckpt = load_pickle(baseline_checkpoint_path(run_dir))
    if ckpt is None:
        return None
    model = build_cnn_model()
    load_model_weights_into(model, ckpt["model_weights_path"])
    return ckpt["round_idx"], model, ckpt["client_state"], ckpt["history"]

def _normalize_client_to_model_map(client_to_model):
    return {int(cid): int(mid) for cid, mid in client_to_model.items()}

def _normalize_cluster_members_map(cluster_members):
    normalized = {}
    for mid, members in cluster_members.items():
        mid = int(mid)
        normalized[mid] = set(int(cid) for cid in members)
    return normalized

def sanitize_feddrift_state(fed_state):
    """
    Make sure models, cluster_members and client_to_model are consistent.

    This is especially important when resuming from an older checkpoint that may
    have been interrupted while saving, or created by a previous notebook version.
    """
    models = {int(mid): model for mid, model in fed_state.get("models", {}).items()}
    cluster_members = _normalize_cluster_members_map(fed_state.get("cluster_members", {}))
    client_to_model = _normalize_client_to_model_map(fed_state.get("client_to_model", {}))

    # Rebuild cluster membership from both sources.
    rebuilt = defaultdict(set)

    # First use cluster_members when the model still exists.
    for mid, members in cluster_members.items():
        if mid in models:
            for cid in members:
                if 0 <= cid < N_CLIENTS:
                    rebuilt[mid].add(cid)

    # Then use client_to_model as a stronger source of truth for client assignment.
    for cid in range(N_CLIENTS):
        mid = client_to_model.get(cid, None)
        if mid is not None and mid in models:
            rebuilt[mid].add(cid)

    # Any client still missing is assigned to the smallest available model.
    if models:
        default_mid = min(models.keys())
        assigned = set()
        for mid, members in rebuilt.items():
            assigned.update(members)
        for cid in range(N_CLIENTS):
            if cid not in assigned:
                rebuilt[default_mid].add(cid)
                client_to_model[cid] = default_mid

    # Remove models that ended up with no members.
    final_models = {}
    final_members = {}
    for mid, model in models.items():
        members = set(rebuilt.get(mid, set()))
        if members:
            final_models[mid] = model
            final_members[mid] = members

    # Fallback if something went badly wrong: start from one cluster.
    if not final_models:
        if models:
            keep_mid = min(models.keys())
            final_models = {keep_mid: models[keep_mid]}
            final_members = {keep_mid: set(range(N_CLIENTS))}
            client_to_model = {cid: keep_mid for cid in range(N_CLIENTS)}
        else:
            raise RuntimeError("FedDrift checkpoint is empty or invalid: no models found.")

    # Rebuild client_to_model from the final members.
    final_client_to_model = {}
    for mid, members in final_members.items():
        for cid in members:
            final_client_to_model[cid] = mid

    # Final consistency check for all clients.
    if len(final_client_to_model) != N_CLIENTS:
        default_mid = min(final_models.keys())
        for cid in range(N_CLIENTS):
            final_client_to_model.setdefault(cid, default_mid)
            final_members.setdefault(default_mid, set()).add(cid)

    fed_state["models"] = final_models
    fed_state["cluster_members"] = final_members
    fed_state["client_to_model"] = final_client_to_model
    fed_state["next_model_id"] = max(final_models.keys(), default=-1) + 1
    return fed_state

def save_algo4_checkpoint(run_dir, round_idx, fed_state, client_state, history):
    fed_state = sanitize_feddrift_state(fed_state)

    model_meta = {
        "round_idx": round_idx,
        "client_state": client_state,
        "history": history,
        "model_ids": list(fed_state["models"].keys()),
        "cluster_members": {str(mid): sorted(list(members)) for mid, members in fed_state["cluster_members"].items()},
        "client_to_model": {str(cid): int(mid) for cid, mid in fed_state["client_to_model"].items()},
        "next_model_id": int(fed_state["next_model_id"]),
    }
    save_pickle(algo4_checkpoint_path(run_dir), model_meta)

    # save each model separately
    model_dir = run_dir / "models"
    for mid, model in fed_state["models"].items():
        save_model_weights(model, model_dir / f"model_{mid}.weights.h5")

    save_json(run_dir / "history.json", history)

def load_algo4_checkpoint(run_dir):
    ckpt = load_pickle(algo4_checkpoint_path(run_dir))
    if ckpt is None:
        return None

    fed_state = {
        "models": {},
        "cluster_members": {int(mid): set(members) for mid, members in ckpt.get("cluster_members", {}).items()},
        "client_to_model": {int(k): int(v) for k, v in ckpt.get("client_to_model", {}).items()},
        "next_model_id": int(ckpt.get("next_model_id", 0)),
    }

    model_ids = [int(mid) for mid in ckpt.get("model_ids", [])]
    model_dir = run_dir / "models"

    loaded_any = False
    for mid in model_ids:
        weight_path = model_dir / f"model_{mid}.weights.h5"
        if not weight_path.exists():
            print(f"[load_algo4_checkpoint] Missing model weights for model {mid}: {weight_path}")
            continue
        model = build_cnn_model()
        load_model_weights_into(model, weight_path)
        fed_state["models"][mid] = model
        loaded_any = True

    if not loaded_any:
        raise RuntimeError(
            f"No model weight files found in {model_dir}. "
            "Delete the experiment folder and rerun from scratch."
        )

    fed_state = sanitize_feddrift_state(fed_state)
    return ckpt["round_idx"], fed_state, ckpt["client_state"], ckpt["history"]

In [10]:

# =========================
# 10. Experiment runners (baseline methods and Algorithm 4)
# =========================

def get_round_candidate_pool(round_idx):
    rng = np.random.default_rng(SEED + 7000 + round_idx)
    size = min(CANDIDATE_POOL_SIZE, N_CLIENTS)
    return rng.choice(np.arange(N_CLIENTS), size=size, replace=False).tolist()

def evaluate_single_model(global_model, round_idx, scenario):
    x_clean, y_clean = x_test, y_test
    x_drift, y_drift = get_drifted_eval_set(round_idx, scenario)

    _, clean_acc = evaluate_model(global_model, x_clean, y_clean)
    _, drift_acc = evaluate_model(global_model, x_drift, y_drift)
    return clean_acc, drift_acc

def run_baseline_experiment(strategy, k, scenario, resume=True):
    run_dir = experiment_dir(strategy, k, scenario)
    ckpt = load_baseline_checkpoint(run_dir) if resume else None

    if ckpt is None:
        start_round = 0
        global_model = build_cnn_model()
        client_state = initialize_client_state(N_CLIENTS)
        history = make_history()
    else:
        start_round, global_model, client_state, history = ckpt
        start_round = int(start_round) + 1
        print(f"[{strategy} | K={k} | {scenario}] Resuming from round {start_round}")

    for r in range(start_round, ROUNDS):
        round_cache = {cid: get_client_round_data(cid, r, scenario) for cid in range(N_CLIENTS)}
        candidate_pool = get_round_candidate_pool(r)

        if r < WARMUP_ROUNDS:
            selected_clients = random_select(candidate_pool, k)
        else:
            selected_clients = select_clients(strategy, candidate_pool, k, client_state)

        global_model, comm_bytes, comp_samples = train_selected_clients_single_model(
            global_model,
            selected_clients,
            round_cache,
            client_state,
            r,
        )

        clean_acc, drift_acc = evaluate_single_model(global_model, r, scenario)

        drift_scores = [client_state[c]["drift_score"] for c in selected_clients] if selected_clients else [0.0]
        proxy_scores = [client_state[c]["proxy_score"] for c in selected_clients] if selected_clients else [0.0]

        history["round"].append(r + 1)
        history["clean_accuracy"].append(float(clean_acc))
        history["drifted_accuracy"].append(float(drift_acc))
        history["communication_bytes"].append(int(comm_bytes))
        history["computation_samples"].append(int(comp_samples))
        history["selected_clients"].append(selected_clients)
        history["cluster_count"].append(1)
        history["drifted_clients_count"].append(int(sum(cs["last_drift_round"] == r for cs in client_state.values())))
        history["mean_drift_score"].append(float(np.mean(np.abs(drift_scores))))
        history["mean_proxy_score"].append(float(np.mean(proxy_scores)))

        save_baseline_checkpoint(run_dir, r, global_model, client_state, history)

        if (r + 1) % 10 == 0 or r == 0:
            print(f"[{strategy} | K={k} | {scenario}] round {r+1:03d} | clean={clean_acc:.4f} | drift={drift_acc:.4f}")

    return global_model, history, client_state

def run_algo4_experiment(k, scenario, resume=True):
    strategy = "feddrift_algo4"
    run_dir = experiment_dir(strategy, k, scenario)
    ckpt = load_algo4_checkpoint(run_dir) if resume else None

    if ckpt is None:
        start_round = 0
        fed_state = initialize_feddrift_state()
        client_state = initialize_client_state(N_CLIENTS)
        history = make_history()
    else:
        start_round, fed_state, client_state, history = ckpt
        start_round = int(start_round) + 1
        print(f"[{strategy} | K={k} | {scenario}] Resuming from round {start_round}")
        fed_state = ensure_cluster_keys_exist(fed_state)

    for r in range(start_round, ROUNDS):
        round_cache = {cid: get_client_round_data(cid, r, scenario) for cid in range(N_CLIENTS)}

        # Algorithm 4 style clustering and merging over all clients
        fed_state, drifted_clients = feddrift_assign_and_merge(fed_state, client_state, round_cache, r)

        # Select K clients from candidate pool using cluster-aware ranking
        candidate_pool = get_round_candidate_pool(r)
        if r < WARMUP_ROUNDS:
            selected_clients = random_select(candidate_pool, k)
        else:
            selected_clients = select_k_clients_from_clusters(fed_state, client_state, candidate_pool, k, r)

        # Train per cluster
        fed_state, comm_bytes, comp_samples = train_selected_clients_per_cluster(
            fed_state,
            selected_clients,
            round_cache,
            client_state,
            r,
        )

        # Evaluate multi-model system
        x_drift, y_drift = get_drifted_eval_set(r, scenario)
        _, clean_acc = evaluate_feddrift_ensemble(fed_state, x_test, y_test)
        _, drift_acc = evaluate_feddrift_ensemble(fed_state, x_drift, y_drift)

        drift_scores = [client_state[c]["drift_score"] for c in selected_clients] if selected_clients else [0.0]
        proxy_scores = [client_state[c]["proxy_score"] for c in selected_clients] if selected_clients else [0.0]

        history["round"].append(r + 1)
        history["clean_accuracy"].append(float(clean_acc))
        history["drifted_accuracy"].append(float(drift_acc))
        history["communication_bytes"].append(int(comm_bytes))
        history["computation_samples"].append(int(comp_samples))
        history["selected_clients"].append(selected_clients)
        history["cluster_count"].append(int(len(fed_state["models"])))
        history["drifted_clients_count"].append(int(len(drifted_clients)))
        history["mean_drift_score"].append(float(np.mean(np.abs(drift_scores))))
        history["mean_proxy_score"].append(float(np.mean(proxy_scores)))

        save_algo4_checkpoint(run_dir, r, fed_state, client_state, history)

        if (r + 1) % 10 == 0 or r == 0:
            print(f"[{strategy} | K={k} | {scenario}] round {r+1:03d} | clusters={len(fed_state['models'])} | clean={clean_acc:.4f} | drift={drift_acc:.4f}")

    return fed_state, history, client_state

def run_experiment(strategy, k, scenario, resume=True):
    if strategy == "feddrift_algo4":
        return run_algo4_experiment(k=k, scenario=scenario, resume=resume)
    if strategy in {"random", "divfl", "sfedavg"}:
        return run_baseline_experiment(strategy=strategy, k=k, scenario=scenario, resume=resume)
    raise ValueError(f"Unknown strategy: {strategy}")


In [ ]:

# =========================
# 11. Run all experiments
# =========================

STRATEGIES = ["feddrift_algo4"]

task3_results = {}

for scenario in SCENARIOS:
    print("\n" + "=" * 100)
    print("SCENARIO:", scenario)
    print("=" * 100)

    for k in K_VALUES:
        for strategy in STRATEGIES:
            model_or_state, history, client_state = run_experiment(
                strategy=strategy,
                k=k,
                scenario=scenario,
                resume=True,
            )

            task3_results[(strategy, k, scenario)] = {
                "history": history,
                "run_dir": str(experiment_dir(strategy, k, scenario)),
            }

print("\nDone. Runs:", len(task3_results))



SCENARIO: label_drift
[feddrift_algo4 | K=10 | label_drift] Resuming from round 30


In [ ]:

# =========================
# 12. Save combined results and summary table
# =========================

combined_results_path = BASE_DIR / "task3_results.pkl"
save_pickle(combined_results_path, task3_results)

summary_rows = []
for (strategy, k, scenario), payload in task3_results.items():
    hist = payload["history"]
    summary_rows.append(history_summary_row(strategy, k, scenario, hist))

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(["scenario", "K", "strategy"]).reset_index(drop=True)
summary_csv_path = BASE_DIR / "task3_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("Saved:", combined_results_path)
print("Saved:", summary_csv_path)
display(summary_df)


In [ ]:

# =========================
# 13. Plot helpers
# =========================

def plot_accuracy_curves(scenario, k):
    plt.figure(figsize=(12, 5))
    for strategy in STRATEGIES:
        key = (strategy, k, scenario)
        if key not in task3_results:
            continue
        hist = task3_results[key]["history"]
        plt.plot(hist["round"], hist["clean_accuracy"], linewidth=2, label=f"{strategy} clean")
        plt.plot(hist["round"], hist["drifted_accuracy"], linestyle="--", linewidth=1.8, alpha=0.85, label=f"{strategy} drifted")

    for boundary in list(LABEL_DRIFT_ROUNDS if scenario == "label_drift" else FEATURE_DRIFT_ROUNDS):
        plt.axvline(boundary, linestyle=":", linewidth=1.2, alpha=0.8)

    plt.title(f"Accuracy vs Round | scenario={scenario} | K={k}")
    plt.xlabel("Federated round")
    plt.ylabel("Accuracy")
    plt.grid(True, alpha=0.3)
    plt.legend(ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()

def plot_cost_curves(scenario, k):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for strategy in STRATEGIES:
        key = (strategy, k, scenario)
        if key not in task3_results:
            continue
        hist = task3_results[key]["history"]
        comm_mb = np.cumsum(hist["communication_bytes"]) / (1024**2)
        comp = np.cumsum(hist["computation_samples"])
        axes[0].plot(hist["round"], comm_mb, linewidth=2, label=strategy)
        axes[1].plot(hist["round"], comp, linewidth=2, label=strategy)

    axes[0].set_title(f"Cumulative Communication (MB) | {scenario} | K={k}")
    axes[1].set_title(f"Cumulative Computation (samples) | {scenario} | K={k}")

    for ax in axes:
        ax.set_xlabel("Federated round")
        ax.grid(True, alpha=0.3)
        ax.legend()

    axes[0].set_ylabel("MB")
    axes[1].set_ylabel("Samples")
    plt.tight_layout()
    plt.show()

def plot_cluster_count(scenario, k):
    key = ("feddrift_algo4", k, scenario)
    if key not in task3_results:
        return
    hist = task3_results[key]["history"]
    plt.figure(figsize=(12, 4.5))
    plt.plot(hist["round"], hist["cluster_count"], linewidth=2)
    for boundary in list(LABEL_DRIFT_ROUNDS if scenario == "label_drift" else FEATURE_DRIFT_ROUNDS):
        plt.axvline(boundary, linestyle=":", linewidth=1.2, alpha=0.8)
    plt.title(f"Algorithm 4 | Number of clusters over time | scenario={scenario} | K={k}")
    plt.xlabel("Federated round")
    plt.ylabel("Cluster count")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_final_bar_chart(scenario, metric="final_drifted_accuracy"):
    df = summary_df[summary_df["scenario"] == scenario].copy()
    if df.empty:
        return
    plt.figure(figsize=(12, 5))
    labels = [f"{row.strategy}-K{row.K}" for row in df.itertuples()]
    values = df[metric].astype(float).values
    plt.bar(labels, values)
    plt.xticks(rotation=35, ha="right")
    plt.title(f"{metric} | scenario={scenario}")
    plt.ylabel(metric)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:

# =========================
# 14. Generate the requested plots
# =========================

for scenario in SCENARIOS:
    for k in K_VALUES:
        plot_accuracy_curves(scenario, k)
        plot_cost_curves(scenario, k)
        if ("feddrift_algo4", k, scenario) in task3_results:
            plot_cluster_count(scenario, k)

for scenario in SCENARIOS:
    plot_final_bar_chart(scenario, metric="final_clean_accuracy")
    plot_final_bar_chart(scenario, metric="final_drifted_accuracy")


In [ ]:

# =========================
# 15. Final summary tables for the report
# =========================

for scenario in SCENARIOS:
    print("\n" + "=" * 120)
    print("SUMMARY:", scenario)
    print("=" * 120)
    display(
        summary_df[summary_df["scenario"] == scenario]
        .sort_values(["K", "strategy"])
        .reset_index(drop=True)
    )
